# Урок 1.7. Эксперименты с промптами: few-shot и chain-of-thought

**Модуль 1 «Введение в LLM» · Курс «LLM для Python-разработчиков»**

В уроке 1.2 мы научились писать структурные промпты, а в 1.6 увидели, что модель — «продолжатель паттернов». Сегодня превратим это в два главных приёма промптинга: **few-shot** (обучение примерами прямо в промпте) и **chain-of-thought** (дать модели подумать вслух). Но устроен урок не как парад побед: мы будем **измерять** каждый приём — и один из экспериментов закончится не так, как обещают учебники. Это и есть настоящий промпт-инжиниринг.

## 🎯 Цели урока

После этого урока вы сможете:

- строить **мини-eval**: размеченный тест-набор + метрика — фундамент любых улучшений промпта;
- применять **few-shot** и — что важнее — понимать, на каких задачах он помогает, а на каких добавляет шум;
- применять **chain-of-thought (CoT)** и объяснять, *почему* токены рассуждений делают модель умнее;
- отделять рассуждения от ответа, чтобы не ломать парсинг;
- знать пределы промптинга — и момент, когда пора думать о fine-tuning (модуль 6).

## План

1. Промптинг — это инженерия: сначала метрика
2. Полигон №1: маршрутизатор тикетов
3. Zero-shot: точка отсчёта
4. Few-shot: проверка интуиции (со спойлером: интуиция врёт)
5. Полигон №2: где few-shot блистает — формат
6. Chain-of-thought: токены как рабочая память
7. Собираем взрослый процесс

⏱ **Время:** ~75–90 минут.

> 💡 Урок полностью живой: нужна Ollama с `qwen2.5:3b` (урок 1.5). Все цифры точности в тексте — из реального прогона; ваши могут немного отличаться, и это нормально: часть выводов вы сделаете по собственным данным.

## 1. Промптинг — это инженерия: сначала метрика

У «промпт-инжиниринга» дурная репутация: интернет полон магических заклинаний («предложи модели чаевые», «скажи, что от ответа зависит твоя карьера»). Мы пойдём другим путём — тем же, каким вы оптимизируете код:

1. **Сначала тест, потом оптимизация.** Размеченный набор примеров и метрика — до любых правок. Иначе «стало лучше» — самообман: на одном примере полегчало, на трёх других сломалось, а вы и не заметили.
2. **Одна переменная за раз.** Добавляем примеры — не трогаем инструкцию. Иначе не узнать, что именно сработало.
3. **Промпт — это код.** Он живёт в git, у него есть версии, ревью и регрессионные тесты — тот самый eval.

Набор размеченных примеров с метрикой называют **eval** — это слово будет преследовать нас до конца курса: точно так же мы будем мерить RAG и fine-tuning.

In [ ]:
import httpx

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen2.5:3b"

try:
    httpx.get(OLLAMA_URL, timeout=2)
    OLLAMA_AVAILABLE = True
    print("Ollama на месте — урок будет полностью живым.")
except (httpx.ConnectError, httpx.TimeoutException):
    OLLAMA_AVAILABLE = False
    print("Ollama недоступна — запустите её (урок 1.5); живые ячейки пропустятся.")

def chat(messages: list, *, temperature: float = 0.0, num_predict: int = 200) -> str | None:
    """Наш стандартный вызов модели. temperature=0 -> жадная генерация:
    одинаковый вход даёт одинаковый выход - то, что нужно для замеров."""
    if not OLLAMA_AVAILABLE:
        print("[пропущено: Ollama недоступна]")
        return None
    r = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL, "messages": messages, "stream": False,
        "options": {"temperature": temperature, "num_predict": num_predict},
    }, timeout=300)
    r.raise_for_status()
    return r.json()["message"]["content"]

## 2. Полигон №1: маршрутизатор тикетов

Первая задача — классика бэкенда: **автоматическая маршрутизация обращений** по четырём очередям: `оплата`, `техника`, `аккаунт`, `другое`. Границы категорий — это бизнес-правила нашей вымышленной компании:

| ситуация | категория |
|---|---|
| тарифы, счета, возвраты, неудачные платежи | `оплата` |
| продукт сломан — даже если сломался экран оплаты | `техника` |
| доступ, пароль, почта, удаление данных | `аккаунт` |
| партнёрства, вакансии, всё остальное | `другое` |

Собираем eval: 12 размеченных обращений, метрика — точное совпадение метки. Заодно напишем универсальные хелперы — они прослужат весь урок:

In [ ]:
TEST_SET = [
    ("Списали деньги дважды за один заказ",                     "оплата"),
    ("Сайт не открывается с телефона, пишет ошибку 500",        "техника"),
    ("Не приходит письмо для сброса пароля",                    "аккаунт"),
    ("Как сменить тарифный план?",                              "оплата"),
    ("Приложение вылетает, когда нажимаю «Оплатить»",           "техника"),
    ("Удалите мой профиль и все данные",                        "аккаунт"),
    ("Хочу вернуть деньги за подписку",                         "оплата"),
    ("Где посмотреть ваши вакансии?",                           "другое"),
    ("После обновления пропала тёмная тема",                    "техника"),
    ("Меня взломали, кто-то поменял почту в профиле",           "аккаунт"),
    ("Выставите, пожалуйста, счёт на юрлицо",                   "оплата"),
    ("Предлагаю партнёрство, с кем можно поговорить?",          "другое"),
]

def normalize(answer: str) -> str:
    """Мягкая нормализация: регистр, пунктуация и префиксы вроде
    'Категория: ...' не должны считаться ошибкой."""
    cleaned = answer.split(":")[-1] if ":" in answer else answer
    return cleaned.strip().lower().strip(".,!«»\"' ")

def ask_with_examples(system_prompt: str, examples: list, text: str,
                      num_predict: int = 10) -> str | None:
    """Запрос с few-shot примерами, оформленными парами сообщений."""
    messages = [{"role": "system", "content": system_prompt}]
    for example_text, label in examples:          # примеры - как история диалога
        messages.append({"role": "user", "content": example_text})
        messages.append({"role": "assistant", "content": label})
    messages.append({"role": "user", "content": text})
    answer = chat(messages, num_predict=num_predict)
    return None if answer is None else normalize(answer)

def run_eval(system_prompt: str, examples: list = (),
             test_set: list = TEST_SET, num_predict: int = 10) -> tuple[float, list]:
    """Прогоняет тест-набор, возвращает (точность, список ошибок)."""
    errors = []
    for text, expected in test_set:
        got = ask_with_examples(system_prompt, list(examples), text, num_predict)
        if got != expected:
            errors.append((text, expected, got))
    accuracy = 1 - len(errors) / len(test_set)
    return accuracy, errors

print(f"Eval готов: {len(TEST_SET)} примеров, метрика — точное совпадение метки")

## 3. Zero-shot: точка отсчёта

**Zero-shot** — просто попросили, без единого примера. Это всегда первый замер: дёшево, быстро — и для многих задач уже достаточно. Промпт пишем добросовестно, по правилам урока 1.2 — роль, задача, формат:

In [ ]:
SYSTEM_PROMPT = (
    "Ты — маршрутизатор обращений в службу поддержки. "
    "Определи категорию обращения пользователя: оплата, техника, аккаунт или другое. "
    "Ответь одним словом — названием категории, строчными буквами."
)

if OLLAMA_AVAILABLE:
    acc_zero, errors_zero = run_eval(SYSTEM_PROMPT)
    print(f"Zero-shot: точность {acc_zero:.0%} "
          f"({len(TEST_SET) - len(errors_zero)}/{len(TEST_SET)})\n")
    for text, expected, got in errors_zero:
        print(f"  ждали {expected!r:10} получили {got!r:12} <- {text}")

В нашем прогоне zero-shot взял **92% (11/12)**, а единственной ошибкой оказался курьёз: на «удалите мой профиль» модель ответила… `akkount` — правильная категория, но транслитом. Классическая **ошибка формата**, которую строгая метрика справедливо засчитала как промах (парсеру в проде тоже будет всё равно, почему строка не совпала).

Вывод из точки отсчёта: категории вроде «оплата/техника/аккаунт» — общечеловеческие, модель хорошо понимает их без подсказок. Запомним это — сейчас пригодится.

## 4. Few-shot: проверка интуиции

**Few-shot** — добавить в промпт несколько разобранных примеров «вход → правильный выход». Почему это должно работать, вы знаете из урока 1.6: модель — продолжатель паттернов. База продолжала начатый список; instruct, увидев серию пар «обращение → метка», продолжает серию в том же формате и по той же логике.

Как оформлять: канонический способ для чат-моделей — **парами сообщений** `user`/`assistant`, как будто диалог уже шёл и модель уже отвечала правильно (наш `ask_with_examples` так и делает). После урока 1.6 вы понимаете, во что это превращает шаблон чата: в документ с образцами.

Интуиция говорит: «примеры всегда помогают». Проверим — системный промпт не меняем ни на букву, добавляем только примеры (одна переменная за раз!):

In [ ]:
FEW_SHOT_EXAMPLES = [
    ("Не могу оплатить картой, платёж отклоняется",        "оплата"),
    ("Сколько стоит тариф Про и как на него перейти?",     "оплата"),
    ("Кнопка оплаты не нажимается, в консоли ошибка JS",   "техника"),
    ("Страница профиля грузится целую минуту",             "техника"),
    ("Как поменять email в настройках профиля?",           "аккаунт"),
    ("Хотим купить рекламу на вашем сайте",                "другое"),
]

if OLLAMA_AVAILABLE:
    acc_few, errors_few = run_eval(SYSTEM_PROMPT, FEW_SHOT_EXAMPLES)
    print(f"Zero-shot: {acc_zero:.0%}   ->   Few-shot: {acc_few:.0%}\n")
    for text, expected, got in errors_few:
        print(f"  ждали {expected!r:10} получили {got!r:12} <- {text}")

Сюрприз. В нашем прогоне few-shot **уронил** точность: 92% → 83%, причём появились ошибки там, где zero-shot отвечал верно («не приходит письмо для сброса пароля» вдруг уехало в `оплата`). Что произошло?

1. **Задача была уже по зубам zero-shot.** Категории общечеловеческие, конвенций почти нет — примерам нечего добавить, кроме шума.
2. **Примеры — это не только сигнал, но и шум.** Шесть пар сообщений меняют «документ», который видит модель; внимание перетягивается на поверхностные совпадения слов с примерами.
3. **Маленький тест-набор шумит.** 92% против 83% на 12 примерах — это разница в одну ошибку. На проде eval из 50–100 примеров отличал бы сигнал от шума надёжнее.

Но главный вывод другой: **eval поймал деградацию.** Если бы мы правили промпт «на глазок», то посмотрели бы на пару удачных ответов и задеплоили ухудшение с чувством выполненного долга. Замер отработал ровно так, как должен.

Значит ли это, что few-shot бесполезен? Нет — мы просто стреляли им не по той мишени.

## 5. Полигон №2: где few-shot блистает — формат

Стихия few-shot — задачи, где важен **точный формат вывода** и **произвольные конвенции**, которые словами описывать долго и без толку. Новый полигон: извлечение полей из описаний заказов в строгую строку `номер;YYYY-MM-DD;сумма;статус`. Инструкция описывает формат добросовестно и подробно — посмотрим, хватит ли этого без примеров:

In [ ]:
FORMAT_PROMPT = (
    "Извлеки из описания заказа четыре поля и выведи одной строкой через \";\" "
    "без пробелов вокруг разделителя: номер заказа (только число), дата в формате "
    "YYYY-MM-DD, сумма в рублях (целое число без пробелов и копеек), статус "
    "строчными буквами. Больше ничего не выводи."
)

FORMAT_TEST_SET = [
    ("Заказ №4821 от 03.05.2026, клиент ООО «Ромашка», сумма 15 200 руб., статус: Оплачен",
     "4821;2026-05-03;15200;оплачен"),
    ("Заказ #77 от 9 января 2026 г., ИП Петров, сумма 990 ₽, статус ОТМЕНЁН",
     "77;2026-01-09;990;отменён"),
    ("заказ № 5512, дата 2026/07/12, ООО Вектор, 48 000,00 руб., статус: в доставке",
     "5512;2026-07-12;48000;в доставке"),
    ("Заказ №306 (12.11.2025). Сумма — 1 499 руб. Статус: Возврат",
     "306;2025-11-12;1499;возврат"),
    ("№901 от 28.02.2026, АО ТехСила, сумма 250 000 руб, статус: оплачен",
     "901;2026-02-28;250000;оплачен"),
    ("Заказ 133 от 5 мая 2026, сумма 75 руб., статус: Ожидает оплаты",
     "133;2026-05-05;75;ожидает оплаты"),
    ("Заказ №2048 от 31.12.2025, клиент Смирнова А.А., сумма 3 210 ₽, статус: отменён",
     "2048;2025-12-31;3210;отменён"),
    ("заказ №64 · 2026-03-01 · 18 500 руб · статус ДОСТАВЛЕН",
     "64;2026-03-01;18500;доставлен"),
]

FORMAT_EXAMPLES = [
    ("Заказ №15 от 02.03.2026, ООО Пример, сумма 1 000 руб., статус: Новый",
     "15;2026-03-02;1000;новый"),
    ("заказ #8 от 7 июля 2025, сумма 500 ₽, статус ЗАКРЫТ",
     "8;2025-07-07;500;закрыт"),
    ("№ 420, дата 2025/10/30, сумма 9 900,00 руб, статус: в обработке",
     "420;2025-10-30;9900;в обработке"),
]

if OLLAMA_AVAILABLE:
    acc_fmt_zero, err_fmt_zero = run_eval(FORMAT_PROMPT, (),
                                          FORMAT_TEST_SET, num_predict=40)
    acc_fmt_few, err_fmt_few = run_eval(FORMAT_PROMPT, FORMAT_EXAMPLES,
                                        FORMAT_TEST_SET, num_predict=40)
    print(f"zero-shot: {acc_fmt_zero:.0%}   few-shot: {acc_fmt_few:.0%}\n")
    print("Перлы zero-shot:")
    for _, expected, got in err_fmt_zero[:4]:
        print(f"  ждали {expected!r}\n  взяли {got!r}\n")
    if err_fmt_few:
        print("Оставшиеся ошибки few-shot:")
        for _, expected, got in err_fmt_few:
            print(f"  ждали {expected!r}\n  взяли {got!r}")

Вот теперь картина из учебника. В нашем прогоне: zero-shot — **38% (3/8)**, few-shot — **88% (7/8)**. И посмотрите на перлы zero-shot: модель теряла разделители (`48212026-05-0315200оплачен`), переставляла дату (`30612-11-2025`), а один статус ухитрилась выдать как `otmenenенbsp;`. Инструкция всё это описывала правильно — но **три примера объяснили формат лучше, чем абзац текста**.

Осталась одна ошибка few-shot, и она поучительна: модель написала «отменен» вместо «отменён». Буква «ё»! Строгая метрика бескомпромиссна — и это правильно: парсер в проде тоже не простит. Лечится либо примером с «ё», либо нормализацией на стороне парсера — решение за вами, но решение это *инженерное*, а не «на глаз».

Мораль двух полигонов: **few-shot — инструмент для формата и конвенций, а не универсальный усилитель.** Помогает там, где модель не может угадать ваши правила; вредит там, где и так всё понимала.

И последний штрих — формат можно задать вообще без слов. Инструкция из двух слов, всю спецификацию несут примеры:

In [ ]:
if OLLAMA_AVAILABLE:
    date_examples = [
        ("5 марта 2024",    "2024-03-05"),
        ("17 августа 1999", "1999-08-17"),
        ("1 января 2026",   "2026-01-01"),
    ]
    for test_date in ["29 февраля 2024", "31 декабря 2025"]:
        result = ask_with_examples("Преобразуй дату.", date_examples, test_date)
        print(f"{test_date:>16}  ->  {result}")

В нашем прогоне вторая дата преобразовалась идеально, а вот на «29 февраля 2024» модель… начала рассуждать про високосный год вместо ответа. Ещё одно честное напоминание: **few-shot задаёт формат, но не гарантирует его**. Поэтому Pydantic-валидация из урока 1.2 не отменяется никогда — промптинг снижает частоту брака, валидация его ловит.

## 6. Chain-of-thought: токены как рабочая память

Теперь другой класс задач — где нужно **посчитать**. Здесь у LLM фундаментальное ограничение, известное нам с урока 1.1: на каждый токен модель тратит **фиксированное количество вычислений** (один проход по слоям — токен готов). Она физически не может «подумать подольше» перед ответом: попросили сразу число — весь расчёт должен уместиться в один проход.

Но есть лазейка, и это снова авторегрессия: каждый следующий токен видит все предыдущие. Разрешим модели писать промежуточные шаги — и она сможет «сохранять» промежуточные результаты в собственный вывод и опираться на них. Как функция, которая логирует переменные и читает свой же лог. Это и есть **chain-of-thought**: рассуждение вслух как внешняя рабочая память.

Меряем, как всегда. Два режима на шести задачах, до боли знакомых бэкендеру. Заметьте: в CoT-режиме **ответ отделён от рассуждений** договорённым маркером «Ответ:» — иначе парсинг превращается в лотерею:

In [ ]:
import re

MATH_TASKS = [
    ("У сервера 16 ГБ RAM. ОС занимает 4 ГБ, каждый контейнер требует 1.5 ГБ. "
     "Сколько контейнеров поместится целиком?", 8),
    ("Батч из 1200 задач разбирают 3 воркера, каждый обрабатывает 25 задач в минуту. "
     "За сколько минут они закончат?", 16),
    ("Из 250 запросов 12% завершились ошибкой. Сколько запросов прошли успешно?", 220),
    ("Диск на 500 ГБ заполнен на 60%. Логи прибавляют 10 ГБ в день. "
     "Через сколько дней диск будет полон?", 20),
    ("API разрешает 90 запросов в минуту. Скрипт шлёт 4 запроса в секунду. "
     "На сколько запросов в минуту скрипт превышает лимит?", 150),
    ("В команде 3 разработчика, каждый пишет код по 6 часов в день. Спринт длится "
     "10 рабочих дней, оценка бэклога — 150 часов. На сколько часов ёмкость "
     "спринта превышает оценку?", 30),
]

DIRECT_PROMPT = ("Реши задачу. В ответе напиши только итоговое число — "
                 "без рассуждений, единиц измерения и пояснений.")
COT_PROMPT = ("Реши задачу, рассуждая шаг за шагом. "
              "В конце отдельной строкой напиши: Ответ: <число>")

def extract_number(text: str) -> float | None:
    """Берёт последнее число из ответа (для CoT это число после 'Ответ:')."""
    matches = re.findall(r"-?\d+(?:[.,]\d+)?", text)
    return float(matches[-1].replace(",", ".")) if matches else None

def solve(task: str, style_prompt: str, num_predict: int) -> float | None:
    answer = chat([{"role": "system", "content": style_prompt},
                   {"role": "user", "content": task}], num_predict=num_predict)
    return None if answer is None else extract_number(answer)

if OLLAMA_AVAILABLE:
    direct_ok = cot_ok = 0
    print(f"{'задача':<9} {'direct':>10} {'CoT':>10}   (правильный ответ)")
    print("-" * 48)
    for i, (task, correct) in enumerate(MATH_TASKS, 1):
        d = solve(task, DIRECT_PROMPT, num_predict=20)
        c = solve(task, COT_PROMPT, num_predict=300)
        direct_ok += (d == correct)
        cot_ok += (c == correct)
        fmt = lambda x: "-" if x is None else f"{x:g} {'+' if x == correct else 'x'}"
        print(f"№{i:<8} {fmt(d):>10} {fmt(c):>10}   ({correct})")
    print("-" * 48)
    print(f"Итого:  direct {direct_ok}/{len(MATH_TASKS)},  CoT {cot_ok}/{len(MATH_TASKS)}")

В нашем прогоне разгром: **direct 1/6, CoT 6/6.** Без права на рассуждения модель угадала только простейшую задачу, с рассуждениями — решила все. Это не магия и не «просто длиннее ответ»: это дополнительные вычисления, размазанные по токенам.

Три предостережения, прежде чем включать CoT везде:

1. **CoT стоит денег и времени.** 300 токенов рассуждений вместо 3 токенов ответа — в десятки раз дороже (вспомните цены output-токенов из урока 1.3) и заметно медленнее.
2. **Красивое рассуждение ≠ правильное.** Модель может убедительно расписать шаги и ошибиться в третьем. CoT повышает вероятность успеха — контроль остаётся за eval.
3. **Рассуждения — не для пользователя.** Показывайте финальный ответ, цепочку — в логи для отладки.

Кстати, современные «думающие» модели (OpenAI o-серия, DeepSeek-R1, Qwen с суффиксом `-thinking`) — это CoT, **вшитый обучением**: модель приучили всегда рассуждать в специальном блоке перед ответом. Идея та же, что вы применили руками.

## 7. Собираем взрослый процесс

Чек-лист улучшения промпта — вешайте на стену:

```text
1. Соберите eval: 10-30+ размеченных примеров + метрика    (у нас: 12 и 8 + accuracy)
2. Замерьте zero-shot — это точка отсчёта
3. Разберите ошибки:
     формат и конвенции?  -> few-shot с примерами на границы
     многошаговый расчёт? -> chain-of-thought с маркером "Ответ:"
     всё и так хорошо?    -> не трогайте, вы закончили
4. Меняйте одну переменную за раз, перемеряйте
5. Промпт + примеры + eval -> в git
6. Упёрлись в потолок? Модель побольше (урок 1.3) или fine-tuning (модуль 6)
```

Пункт 3 с вариантом «не трогайте» — не шутка: вы сегодня видели, как непрошеная оптимизация роняет метрику.

## ⚠️ Частые ошибки

1. **Улучшать промпт без eval.** «Вроде стало лучше» на двух примерах — так рождаются промпты-франкенштейны, которые страшно трогать. Сначала тест-набор.
2. **Считать few-shot универсальным усилителем.** На задачах, где zero-shot уже справляется, примеры добавляют шум — точность может *упасть* (вы это видели). Единственный арбитр — замер.
3. **Ошибки и разнобой в примерах.** Модель копирует паттерн целиком: перепутанная метка, лишняя точка, другой регистр — всё будет честно воспроизведено (упражнение 3).
4. **Примеры только на очевидные случаи.** Очевидное модель умеет и так — примеры нужны на *границы* и *формат*, где живут ваши конвенции.
5. **CoT вперемешку с ответом.** Без маркера «Ответ:» вы будете парсить число из середины рассуждений. Отделяйте всегда.
6. **CoT на простых задачах.** Классификация тикета не требует трёхсот токенов раздумий — это в десятки раз дороже ради нуля выигрыша.
7. **Десятки примеров в промпте.** Каждый пример едет в **каждом** запросе (input-токены, урок 1.3) и приближает предел контекста (урок 1.5). 3–6 прицельных примеров почти всегда бьют 20 посредственных.

## 🏋️ Упражнения

### Упражнение 1 — реабилитация few-shot 🔬

Мы видели, что на маршрутизации few-shot парами сообщений просел. Проведите эксперимент: подайте те же 6 примеров **одним текстовым блоком внутри системного промпта** (формат «Обращение: … → Категория: …») и перемерьте. Сравните три числа: zero-shot, few-shot парами, few-shot блоком. Любой исход — результат: вы либо нашли рабочий вариант подачи, либо доказали, что этой задаче примеры не нужны.

### Упражнение 2 — self-consistency 🗳

CoT с `temperature=0` даёт один маршрут рассуждений — не всегда удачный. Приём **self-consistency**: прогнать CoT 5 раз с `temperature=0.9` и выбрать **мажоритарный** ответ (`collections.Counter`). Примените к любой задаче из `MATH_TASKS` и сравните с одиночным прогоном. Оцените цену приёма в токенах.

### Упражнение 3 — отравленные примеры ☠️

Докажите, что примеры сильнее инструкций: в `FEW_SHOT_EXAMPLES` поменяйте местами метки `оплата` ↔ `техника`, **не трогая системный промпт** (он по-прежнему описывает категории правильно). Прогоните eval маршрутизации и посмотрите, чему модель поверила — регламенту или образцам.

## 📝 Мини-квиз

**1. Почему few-shot способен передать конвенции, которые трудно описать словами?**

<details><summary>Ответ</summary>

Модель — продолжатель паттернов (урок 1.6): увидев серию пар «вход → выход», она продолжает серию, воспроизводя формат и логику примеров. Показать оказывается эффективнее, чем описать, — особенно для формата вывода.

</details>

**2. Инструкция говорит одно, примеры показывают другое. Кто победит?**

<details><summary>Ответ</summary>

Чаще побеждают примеры — модель копирует паттерн, а не сверяется с регламентом. Следствие: качество примеров критично, одна перепутанная метка отравляет поведение (упражнение 3).

</details>

**3. Почему «рассуждай шаг за шагом» реально улучшает арифметику, а не просто удлиняет ответ?**

<details><summary>Ответ</summary>

На каждый токен модель тратит фиксированное количество вычислений. Токены рассуждений — это дополнительные проходы и внешняя «рабочая память»: благодаря авторегрессии модель опирается на уже выписанные промежуточные результаты.

</details>

**4. Когда few-shot может ухудшить результат?**

<details><summary>Ответ</summary>

Когда задача и так по зубам zero-shot: примерам нечего добавить, кроме шума, — внимание модели перетягивается на поверхностные совпадения с примерами. Поэтому любое добавление примеров проверяется замером.

</details>

**5. Как правильно проверить, что новый промпт лучше старого?**

<details><summary>Ответ</summary>

Только замером на фиксированном размеченном наборе с фиксированной метрикой, меняя одну переменную за раз. «Попробовал на паре примеров — вроде лучше» проверкой не является.

</details>

## ✅ Решения упражнений

### Решение 1 — реабилитация few-shot

In [ ]:
if OLLAMA_AVAILABLE:
    examples_block = "\n".join(f"Обращение: {text} -> Категория: {label}"
                                for text, label in FEW_SHOT_EXAMPLES)
    SYSTEM_WITH_BLOCK = SYSTEM_PROMPT + "\n\nПримеры:\n" + examples_block

    acc_block, errors_block = run_eval(SYSTEM_WITH_BLOCK)
    print(f"zero-shot:                 {acc_zero:.0%}")
    print(f"few-shot парами сообщений: {acc_few:.0%}")
    print(f"few-shot блоком в system:  {acc_block:.0%}")
    if errors_block:
        print("\nОшибки блочного варианта:")
        for text, expected, got in errors_block:
            print(f"  ждали {expected!r:10} получили {got!r:12} <- {text}")

В нашем прогоне случился финальный поворот сюжета: **блок в системном промпте дал 100%** — против 92% у zero-shot и 83% у пар сообщений. Few-shot реабилитирован: проседали не примеры, а способ их подачи. После урока 1.6 можно даже предположить почему: пары сообщений выглядят для модели как «прошлые ходы диалога» и конкурируют с реальным вопросом, а блок в системном промпте — как справочник, на который надо опираться.

Два принципа на память: **способ подачи примеров — тоже переменная**, которую надо мерить; и если бы ни один вариант не побил zero-shot, честным итогом было бы «этой задаче примеры не нужны» — знание, когда инструмент *не* нужен, экономит токены в каждом запросе. У вас числа могут отличаться — доверяйте своему замеру, а не нашему тексту.

### Решение 2 — self-consistency

In [ ]:
from collections import Counter

if OLLAMA_AVAILABLE:
    task, correct = MATH_TASKS[3]   # диск и логи
    print("Задача:", task, f"\nПравильный ответ: {correct}\n")

    votes = []
    for _ in range(5):
        answer = chat([{"role": "system", "content": COT_PROMPT},
                       {"role": "user", "content": task}],
                      temperature=0.9, num_predict=300)
        votes.append(extract_number(answer))
    counter = Counter(votes)
    winner, count = counter.most_common(1)[0]

    print("Пять прогонов дали:", votes)
    print(f"Мажоритарный ответ: {winner:g} ({count} из 5) — "
          f"{'верно' if winner == correct else 'неверно'}")
    print("\nЦена: ~5x токенов и времени относительно одиночного CoT.")

Self-consistency — способ купить надёжность за токены: разные маршруты рассуждений ошибаются в разных местах, голосование гасит случайные сбои. Если у вас все пять прогонов сошлись — модель уверенно решает эту задачу, и приём не нужен; его включают точечно, для решений с высокой ценой ошибки.

### Решение 3 — отравленные примеры

In [ ]:
if OLLAMA_AVAILABLE:
    SWAP = {"оплата": "техника", "техника": "оплата"}
    POISONED_EXAMPLES = [(text, SWAP.get(label, label))
                         for text, label in FEW_SHOT_EXAMPLES]

    acc_poisoned, errors_poisoned = run_eval(SYSTEM_PROMPT, POISONED_EXAMPLES)
    print(f"честные примеры:      {acc_few:.0%}")
    print(f"отравленные примеры:  {acc_poisoned:.0%}\n")
    for text, expected, got in errors_poisoned:
        print(f"  ждали {expected!r:10} получили {got!r:12} <- {text}")

В нашем прогоне точность упала до 67%, и посмотрите **как**: «приложение вылетает при оплате» уехало в `оплата`, «счёт на юрлицо» — в `техника`. Модель выучила перевёрнутую границу из примеров, хотя системный промпт всё это время честно описывал правильные категории. Инструкция проиграла образцам.

Практические следствия: примеры для few-shot проходят ревью как код; а если они собираются автоматически (из базы, из истории тикетов) — качество этого источника надо контролировать, иначе один кривой поток данных незаметно перепрограммирует сервис.

## 🎓 Итоги

- **Сначала eval, потом промпт**: размеченный набор + метрика + «одна переменная за раз». Наш eval сегодня дважды спас нас от ложных выводов.
- **Zero-shot** — обязательная точка отсчёта; на общечеловеческих задачах он сильнее, чем принято думать.
- **Few-shot** — инструмент для **формата и конвенций** (у нас: 38% → 88%), а не универсальный усилитель: там, где zero-shot справляется, примеры могут навредить (у нас: 92% → 83%). Примеры сильнее инструкций — это сила и уязвимость сразу.
- **Chain-of-thought** даёт модели дополнительные вычисления токенами рассуждений (у нас: 1/6 → 6/6); ответ отделяйте маркером; на простых задачах — не включать.
- **Self-consistency** — надёжность в обмен на пятикратные токены, точечно.
- Потолок промптинга существует: дальше — модель сильнее или fine-tuning (модуль 6).

**Дальше — урок 1.8 «Закрепление материала»:** финальный квиз по всему модулю и первый настоящий проект — CLI-утилита на Ollama, собранная из всего, что вы узнали.

## 📚 Что почитать

- [Language Models are Few-Shot Learners (GPT-3)](https://arxiv.org/abs/2005.14165) — статья, с которой начался few-shot
- [Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903) — оригинальная статья про CoT
- [Self-Consistency](https://arxiv.org/abs/2203.11171) — голосование цепочек рассуждений
- [Prompt Engineering Guide](https://www.promptingguide.ai/ru) — большой справочник приёмов (есть русская версия)
- [Anthropic: prompt engineering](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview) — практичный вендорский гайд, приёмы переносимы на любые модели